In [1]:
import pandas as pd

orders = pd.read_csv('olist_orders_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
translation = pd.read_csv('product_category_name_translation.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')

In [3]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [5]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

In [6]:
orders_clean = orders[orders['order_status'] == 'delivered'].copy()

In [7]:
orders_clean['real_delivery_day'] = (orders_clean['order_delivered_customer_date'] - orders_clean['order_purchase_timestamp']).dt.days

In [8]:
orders_clean['estimated_delivery_day'] = (orders_clean['order_estimated_delivery_date'] - orders_clean['order_purchase_timestamp']).dt.days

In [9]:
orders_clean['difference_days'] = (orders_clean['estimated_delivery_day'] - orders_clean['real_delivery_day'])

In [11]:
orders_clean[['real_delivery_day', 'estimated_delivery_day', 'difference_days']].head(3)

,real_delivery_day,estimated_delivery_day,difference_days
0,8.0,15,7.0
1,13.0,19,6.0
2,9.0,26,17.0


In [12]:
orders_geo = pd.merge(orders_clean, customers[['customer_id', 'customer_state']], on='customer_id', how='inner')

In [14]:
orders_geo.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,real_delivery_day,estimated_delivery_day,difference_days,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,15,7.0,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,19,6.0,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,26,17.0,GO


In [16]:
orders_geo['real_delivery_day'].mean()

np.float64(12.093604229294082)

In [17]:
orders_geo['difference_days'].mean()

np.float64(11.279143775266922)

In [19]:
orders_geo[['real_delivery_day', 'estimated_delivery_day', 'difference_days']].describe()

,real_delivery_day,estimated_delivery_day,difference_days
count,96470.000000,96478.000000,96470.000000
mean,12.093604,23.372759,11.279144
std,9.551380,8.758137,10.192137
min,0.000000,2.000000,-189.000000
25%,6.000000,18.000000,7.000000
50%,10.000000,23.000000,12.000000
75%,15.000000,28.000000,16.000000
max,209.000000,155.000000,146.000000


In [21]:
orders_geo.groupby('customer_state')['real_delivery_day'].mean().sort_values(ascending=False).head(5)

customer_state
RR    28.975610
AP    26.731343
AM    25.986207
AL    24.040302
PA    23.316068
Name: real_delivery_day, dtype: float64

In [23]:
orders_geo.groupby('customer_state')['real_delivery_day'].mean().sort_values(ascending=True).head(5)


customer_state
SP     8.298094
PR    11.526711
MG    11.542188
DF    12.509135
SC    14.475183
Name: real_delivery_day, dtype: float64

In [25]:
promedio_por_estado = orders_geo.groupby('customer_state')['real_delivery_day'].mean().round(1).sort_values()


In [26]:
promedio_por_estado.head(5)

customer_state
SP     8.3
MG    11.5
PR    11.5
DF    12.5
SC    14.5
Name: real_delivery_day, dtype: float64

In [30]:
df_reviews = order_items.merge(products, on = 'product_id').merge(reviews, on = 'order_id')

In [31]:
cate_summary = df_reviews.groupby('product_category_name')['review_score'].agg(['mean', 'count'])

In [35]:
cate_insatisfaccion = cate_summary[cate_summary['count'] > 50].sort_values(by='mean', ascending=True).head(10)


In [36]:
cate_insatisfaccion.head(10)

,mean,count
product_category_name,,
moveis_escritorio,3.493183,1687
fashion_roupa_masculina,3.641221,131
telefonia_fixa,3.683206,262
audio,3.825485,361
casa_conforto,3.829885,435
construcao_ferramentas_seguranca,3.844560,193
cama_mesa_banho,3.895663,11137
moveis_decoracao,3.903493,8331
moveis_sala,3.904382,502


In [38]:
cate_insatisfaccion['mean'] = cate_insatisfaccion['mean'].round(2)

In [39]:
cate_insatisfaccion

,mean,count
product_category_name,,
moveis_escritorio,3.49,1687
fashion_roupa_masculina,3.64,131
telefonia_fixa,3.68,262
audio,3.83,361
casa_conforto,3.83,435
construcao_ferramentas_seguranca,3.84,193
cama_mesa_banho,3.90,11137
moveis_decoracao,3.90,8331
moveis_sala,3.90,502


In [49]:
import os

In [50]:
os.makedirs('marts', exist_ok=True)

In [51]:
fact_orders = orders_clean[[
    'order_id', 'customer_id', 'order_status',
    'real_delivery_day', 'estimated_delivery_day', 'difference_days'
]].copy()

In [52]:
fact_orders. head(5)

,order_id,customer_id,order_status,real_delivery_day,estimated_delivery_day,difference_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,8.0,15,7.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,13.0,19,6.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,9.0,26,17.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,13.0,26,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2.0,12,10.0


In [53]:
dim_customers = customers[['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']].copy()

In [54]:
dim_customers.head(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [55]:
fact_orders.to_csv('marts/fact_orders.csv', index=False)

In [58]:
dim_customers.to_csv('marts/dim_customers.csv', index=False)

In [56]:
print("--- FACT ORDERS ---")
display(fact_orders.head(3))

--- FACT ORDERS ---


,order_id,customer_id,order_status,real_delivery_day,estimated_delivery_day,difference_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,8.0,15,7.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,13.0,19,6.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,9.0,26,17.0


In [57]:
print("--- DIM CUSTOMERS ---")
display(dim_customers.head(3))

--- DIM CUSTOMERS ---


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
